In [1]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

RANDOM_STATE = 42

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
checkpoint_path = DATA_DIR / "GSE25055_pre_lasso.joblib"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path.resolve()}")

checkpoint = joblib.load(checkpoint_path)
X = checkpoint["X"]
y = checkpoint["y"]

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(outer_cv.split(X, y))

def create_lasso_pipeline():

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "lasso",
            LogisticRegression(
                l1_ratio=1.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=10000,
                random_state=RANDOM_STATE
            )
        )
    ])

parameter_grid = {
    "lasso__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3
    ]
}

print("X:", X.shape, "| y:", y.shape, "| outer folds:", len(outer_splits))

Project root added to path: d:\diplom-project
X: (306, 22283) | y: (306,) | outer folds: 10


In [2]:
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from src.models.custom_random_forest import (
    build_forest,
    predict_forest
)

NUMBER_OF_TREES = 30
MAX_DEPTH = 6
MIN_SAMPLES_SPLIT = 5
CLASSIFICATION_THRESHOLD = 0.30
FOLDS_TO_RUN = 10


def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan
    }

feature_names = X.columns.to_numpy()

fold_records = []
selected_probes_by_fold = {}

custom_oof_probability = pd.Series(np.nan, index=y.index)
custom_oof_prediction = pd.Series(pd.NA, index=y.index, dtype="Int64")
sklearn_oof_probability = pd.Series(np.nan, index=y.index)
sklearn_oof_prediction = pd.Series(pd.NA, index=y.index, dtype="Int64")

experiment_start = time.time()

for fold, (train_idx, val_idx) in enumerate(outer_splits, start=1):
    if fold > FOLDS_TO_RUN:
        break

    print(f"\n=== Fold {fold}/{FOLDS_TO_RUN} ===")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # --- LASSO selection, само върху train частта ---
    search = GridSearchCV(
        estimator=create_lasso_pipeline(),
        param_grid=parameter_grid,
        scoring="average_precision",
        cv=inner_cv, refit=True, n_jobs=-1
    )
    search.fit(X_train, y_train)

    coefficients = search.best_estimator_.named_steps["lasso"].coef_.ravel()
    selected_mask = np.abs(coefficients) > 1e-10
    selected_probes = feature_names[selected_mask]
    selected_probes_by_fold[fold] = list(selected_probes)

    print(f"LASSO избра {len(selected_probes)} probes (C={search.best_params_['lasso__C']})")

    X_train_sel = X_train[selected_probes].to_numpy(dtype=float)
    X_val_sel = X_val[selected_probes].to_numpy(dtype=float)
    y_train_arr = y_train.to_numpy(dtype=int)

    # --- Custom Random Forest ---
    t0 = time.time()
    forest = build_forest(
    X_train_sel,
    y_train_arr,
    number_of_trees=NUMBER_OF_TREES,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    random_state=RANDOM_STATE + fold,
    class_weight="balanced"
)
    custom_pred, custom_proba = predict_forest(forest, X_val_sel, classification_threshold=CLASSIFICATION_THRESHOLD)
    custom_time = time.time() - t0

    custom_oof_probability.loc[X_val.index] = custom_proba
    custom_oof_prediction.loc[X_val.index] = custom_pred

    # --- Sklearn Random Forest (същите избрани probes, честно сравнение) ---
    t0 = time.time()
    sk_model = RandomForestClassifier(
    n_estimators=NUMBER_OF_TREES,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced",
    random_state=RANDOM_STATE + fold,
    n_jobs=-1
)
    sk_model.fit(X_train_sel, y_train_arr)
    sklearn_proba = sk_model.predict_proba(X_val_sel)[:, 1]
    sklearn_pred = (
    sklearn_proba >= CLASSIFICATION_THRESHOLD
).astype(int)
    sklearn_time = time.time() - t0

    sklearn_oof_probability.loc[X_val.index] = sklearn_proba
    sklearn_oof_prediction.loc[X_val.index] = sklearn_pred

    # --- Метрики и за двата модела ---
    custom_metrics = calculate_metrics(y_val.to_numpy(), custom_pred, custom_proba)
    sklearn_metrics = calculate_metrics(y_val.to_numpy(), sklearn_pred, sklearn_proba)

    fold_records.append({"fold": fold, "model": "Custom Random Forest", "selected_probes": len(selected_probes), "rf_time_seconds": custom_time, **custom_metrics})
    fold_records.append({"fold": fold, "model": "Sklearn Random Forest", "selected_probes": len(selected_probes), "rf_time_seconds": sklearn_time, **sklearn_metrics})

    print(f"Custom  ROC-AUC={custom_metrics['roc_auc']:.3f} | Sensitivity={custom_metrics['sensitivity']:.3f} | {custom_time:.1f}s")
    print(f"Sklearn ROC-AUC={sklearn_metrics['roc_auc']:.3f} | Sensitivity={sklearn_metrics['sensitivity']:.3f} | {sklearn_time:.1f}s")

experiment_time = time.time() - experiment_start
print(f"\nОбщо време: {experiment_time:.1f}s")


=== Fold 1/10 ===
LASSO избра 84 probes (C=0.1)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.867 | Sensitivity=0.500 | 13.4s
Sklearn ROC-AUC=0.773 | Sensitivity=0.667 | 0.1s

=== Fold 2/10 ===
LASSO избра 12 probes (C=0.03)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tr

In [6]:
RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "lasso_custom_rf_fold1_threshold_030_pilot"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Метрики за двата модела
fold_results = pd.DataFrame(fold_records)

fold_results.to_csv(
    RESULTS_DIR / "fold1_metrics.csv",
    index=False
)

# Прогноза за всеки пациент от validation частта
fold_predictions = pd.DataFrame({
    "patient": X_val.index,
    "actual_class": y_val.to_numpy(),
    "custom_prediction": custom_pred,
    "custom_probability": custom_proba,
    "sklearn_prediction": sklearn_pred,
    "sklearn_probability": sklearn_proba
})

fold_predictions.to_csv(
    RESULTS_DIR / "fold1_predictions.csv",
    index=False
)

# Избраните от LASSO probes
selected_probes_table = pd.DataFrame({
    "probe_id": selected_probes
})

selected_probes_table.to_csv(
    RESULTS_DIR / "fold1_selected_probes.csv",
    index=False
)

# Пълна информация за експеримента
experiment_data = {
    "experiment_type": "pilot_fold_1",
    "fold": 1,
    "lasso_best_parameters": search.best_params_,
    "selected_probes": list(selected_probes),
    "number_of_trees": NUMBER_OF_TREES,
    "max_depth": MAX_DEPTH,
    "min_samples_split": MIN_SAMPLES_SPLIT,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "class_weight": "balanced",
    "custom_metrics": custom_metrics,
    "sklearn_metrics": sklearn_metrics,
    "custom_forest": forest,
    "sklearn_forest": sk_model
}

joblib.dump(
    experiment_data,
    RESULTS_DIR / "fold1_experiment.joblib"
)

print("Резултатите са запазени в:")
print(RESULTS_DIR.resolve())

Резултатите са запазени в:
D:\diplom-project\results\lasso_custom_rf_fold1_threshold_030_pilot


In [5]:
fold_results = pd.DataFrame(fold_records)
display(fold_results)

comparison_table = (
    fold_results
    .pivot(index="fold", columns="model", values=["roc_auc", "sensitivity", "specificity", "rf_time_seconds"])
    .round(3)
)

display(comparison_table)

,fold,model,selected_probes,rf_time_seconds,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity
0,1,Custom Random Forest,84,13.447926,0.866667,0.726007,0.838710,0.710000,0.545455,0.600000,0.500000,0.920000
1,1,Sklearn Random Forest,84,0.067012,0.773333,0.460996,0.741935,0.713333,0.500000,0.400000,0.666667,0.760000
2,2,Custom Random Forest,12,4.698115,0.766667,0.563675,0.806452,0.626667,0.400000,0.500000,0.333333,0.920000
3,2,Sklearn Random Forest,12,0.061763,0.753333,0.475763,0.806452,0.690000,0.500000,0.500000,0.500000,0.880000
4,3,Custom Random Forest,17,5.935588,0.740000,0.529694,0.741935,0.713333,0.500000,0.400000,0.666667,0.760000
5,3,Sklearn Random Forest,17,0.060975,0.700000,0.567460,0.612903,0.633333,0.400000,0.285714,0.666667,0.600000
6,4,Custom Random Forest,14,4.585315,0.653333,0.278093,0.677419,0.610000,0.375000,0.300000,0.500000,0.720000
7,4,Sklearn Random Forest,14,0.094827,0.553333,0.228421,0.548387,0.466667,0.222222,0.166667,0.333333,0.600000
8,5,Custom Random Forest,15,4.589519,0.846667,0.664352,0.709677,0.756667,0.526316,0.384615,0.833333,0.680000
9,5,Sklearn Random Forest,15,0.113428,0.846667,0.640873,0.677419,0.800000,0.545455,0.375000,1.000000,0.600000


roc_auc                                sensitivity  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                                                    
1                    0.867                 0.773                0.500   
2                    0.767                 0.753                0.333   
3                    0.740                 0.700                0.667   
4                    0.653                 0.553                0.500   
5                    0.847                 0.847                0.833   
6                    0.780                 0.773                0.667   
7                    0.824                 0.824                0.600   
8                    0.668                 0.728                0.400   
9                    0.912                 0.904                0.600   
10                   0.715                 0.708                0.667   

                                     specificity                        \
model Sklearn Random Forest Custom Random Forest Sklearn Random Forest   
fold                                                                     
1                     0.667                0.920                  0.76   
2                     0.500                0.920                  0.88   
3                     0.667                0.760                  0.60   
4                     0.333                0.720                  0.60   
5                     1.000                0.680                  0.60   
6                     0.833                0.760                  0.48   
7                     1.000                0.720                  0.64   
8                     0.600                0.880                  0.64   
9                     1.000                0.840                  0.80   
10                    0.833                0.792                  0.75   

           rf_time_seconds                        
model Custom Random Forest Sklearn Random Forest  
fold                                              
1                   13.448                 0.067  
2                    4.698                 0.062  
3                    5.936                 0.061  
4                    4.585                 0.095  
5                    4.590                 0.113  
6                    4.718                 0.057  
7                   14.620                 0.059  
8                   17.140                 0.126  
9                    6.467                 0.096  
10                   6.364                 0.082

In [6]:
summary = fold_results.pivot(index="fold", columns="model", values="roc_auc").round(3)
summary.columns = ["Custom ROC-AUC", "Sklearn ROC-AUC"]

summary["Custom Sens."] = fold_results.pivot(index="fold", columns="model", values="sensitivity")["Custom Random Forest"].round(3)
summary["Sklearn Sens."] = fold_results.pivot(index="fold", columns="model", values="sensitivity")["Sklearn Random Forest"].round(3)
summary["Custom time (s)"] = fold_results.pivot(index="fold", columns="model", values="rf_time_seconds")["Custom Random Forest"].round(1)
summary["Sklearn time (s)"] = fold_results.pivot(index="fold", columns="model", values="rf_time_seconds")["Sklearn Random Forest"].round(1)

display(summary)

,Custom ROC-AUC,Sklearn ROC-AUC,Custom Sens.,Sklearn Sens.,Custom time (s),Sklearn time (s)
fold,,,,,,
1,0.867,0.773,0.500,0.667,13.4,0.1
2,0.767,0.753,0.333,0.500,4.7,0.1
3,0.740,0.700,0.667,0.667,5.9,0.1
4,0.653,0.553,0.500,0.333,4.6,0.1
5,0.847,0.847,0.833,1.000,4.6,0.1
6,0.780,0.773,0.667,0.833,4.7,0.1
7,0.824,0.824,0.600,1.000,14.6,0.1
8,0.668,0.728,0.400,0.600,17.1,0.1
9,0.912,0.904,0.600,1.000,6.5,0.1


In [7]:
full_comparison = fold_results.pivot(
    index="fold",
    columns="model",
    values=["roc_auc", "average_precision", "f1", "sensitivity", "specificity", "balanced_accuracy"]
).round(3)

display(full_comparison)

roc_auc                          average_precision  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                                                    
1                    0.867                 0.773                0.726   
2                    0.767                 0.753                0.564   
3                    0.740                 0.700                0.530   
4                    0.653                 0.553                0.278   
5                    0.847                 0.847                0.664   
6                    0.780                 0.773                0.447   
7                    0.824                 0.824                0.508   
8                    0.668                 0.728                0.553   
9                    0.912                 0.904                0.711   
10                   0.715                 0.708                0.358   

                                              f1                        \
model Sklearn Random Forest Custom Random Forest Sklearn Random Forest   
fold                                                                     
1                     0.461                0.545                 0.500   
2                     0.476                0.400                 0.500   
3                     0.567                0.500                 0.400   
4                     0.228                0.375                 0.222   
5                     0.641                0.526                 0.545   
6                     0.549                0.500                 0.417   
7                     0.437                0.400                 0.526   
8                     0.612                0.400                 0.353   
9                     0.526                0.500                 0.667   
10                    0.420                0.533                 0.588   

               sensitivity                                specificity  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                                                    
1                    0.500                 0.667                0.920   
2                    0.333                 0.500                0.920   
3                    0.667                 0.667                0.760   
4                    0.500                 0.333                0.720   
5                    0.833                 1.000                0.680   
6                    0.667                 0.833                0.760   
7                    0.600                 1.000                0.720   
8                    0.400                 0.600                0.880   
9                    0.600                 1.000                0.840   
10                   0.667                 0.833                0.792   

                               balanced_accuracy                        
model Sklearn Random Forest Custom Random Forest Sklearn Random Forest  
fold                                                                    
1                      0.76                0.710                 0.713  
2                      0.88                0.627                 0.690  
3                      0.60                0.713                 0.633  
4                      0.60                0.610                 0.467  
5                      0.60                0.757                 0.800  
6                      0.48                0.713                 0.657  
7                      0.64                0.660                 0.820  
8                      0.64                0.640                 0.620  
9                      0.80                0.720                 0.900  
10                     0.75                0.729                 0.792

In [8]:
full_comparison = fold_results.pivot(
    index="fold",
    columns="model",
    values=["roc_auc", "average_precision", "accuracy", "balanced_accuracy", "f1", "precision", "sensitivity", "specificity"]
).round(3)

display(full_comparison)

roc_auc                          average_precision  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                                                    
1                    0.867                 0.773                0.726   
2                    0.767                 0.753                0.564   
3                    0.740                 0.700                0.530   
4                    0.653                 0.553                0.278   
5                    0.847                 0.847                0.664   
6                    0.780                 0.773                0.447   
7                    0.824                 0.824                0.508   
8                    0.668                 0.728                0.553   
9                    0.912                 0.904                0.711   
10                   0.715                 0.708                0.358   

                                        accuracy                        \
model Sklearn Random Forest Custom Random Forest Sklearn Random Forest   
fold                                                                     
1                     0.461                0.839                 0.742   
2                     0.476                0.806                 0.806   
3                     0.567                0.742                 0.613   
4                     0.228                0.677                 0.548   
5                     0.641                0.710                 0.677   
6                     0.549                0.742                 0.548   
7                     0.437                0.700                 0.700   
8                     0.612                0.800                 0.633   
9                     0.526                0.800                 0.833   
10                    0.420                0.767                 0.767   

         balanced_accuracy                                         f1  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                                                    
1                    0.710                 0.713                0.545   
2                    0.627                 0.690                0.400   
3                    0.713                 0.633                0.500   
4                    0.610                 0.467                0.375   
5                    0.757                 0.800                0.526   
6                    0.713                 0.657                0.500   
7                    0.660                 0.820                0.400   
8                    0.640                 0.620                0.400   
9                    0.720                 0.900                0.500   
10                   0.729                 0.792                0.533   

                                       precision                        \
model Sklearn Random Forest Custom Random Forest Sklearn Random Forest   
fold                                                                     
1                     0.500                0.600                 0.400   
2                     0.500                0.500                 0.500   
3                     0.400                0.400                 0.286   
4                     0.222                0.300                 0.167   
5                     0.545                0.385                 0.375   
6                     0.417                0.400                 0.278   
7                     0.526                0.300                 0.357   
8                     0.353                0.400                 0.250   
9                     0.667                0.429                 0.500   
10                    0.588                0.444                 0.455   

               sensitivity                                specificity  \
model Custom Random Forest Sklearn Random Forest Custom Random Forest   
fold                                           

In [ ]:
RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "lasso_balanced_rf"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# 1. Резултати за всеки fold и модел
fold_results = pd.DataFrame(fold_records)

fold_results.to_csv(
    RESULTS_DIR / "fold_results.csv",
    index=False
)


# 2. OOF прогнози за всички пациенти
valid_mask = (
    custom_oof_probability.notna()
    & sklearn_oof_probability.notna()
)

oof_predictions = pd.DataFrame({
    "patient": y.loc[valid_mask].index,
    "actual_class": y.loc[valid_mask].to_numpy(dtype=int),
    "custom_prediction": custom_oof_prediction.loc[
        valid_mask
    ].to_numpy(dtype=int),
    "custom_probability": custom_oof_probability.loc[
        valid_mask
    ].to_numpy(dtype=float),
    "sklearn_prediction": sklearn_oof_prediction.loc[
        valid_mask
    ].to_numpy(dtype=int),
    "sklearn_probability": sklearn_oof_probability.loc[
        valid_mask
    ].to_numpy(dtype=float)
})

oof_predictions.to_csv(
    RESULTS_DIR / "oof_predictions.csv",
    index=False
)


# 3. Обща сравнителна таблица
y_oof = oof_predictions["actual_class"].to_numpy()

custom_overall = calculate_metrics(
    y_oof,
    oof_predictions["custom_prediction"].to_numpy(),
    oof_predictions["custom_probability"].to_numpy()
)

sklearn_overall = calculate_metrics(
    y_oof,
    oof_predictions["sklearn_prediction"].to_numpy(),
    oof_predictions["sklearn_probability"].to_numpy()
)

comparison_table = pd.DataFrame([
    {
        "model": "Custom Random Forest",
        **custom_overall
    },
    {
        "model": "Sklearn Random Forest",
        **sklearn_overall
    }
])

comparison_table.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)


# 4. LASSO probes, избрани във всеки fold
selected_probes_table = pd.DataFrame([
    {
        "fold": fold,
        "probe_id": probe
    }
    for fold, probes in selected_probes_by_fold.items()
    for probe in probes
])

selected_probes_table.to_csv(
    RESULTS_DIR / "selected_probes_by_fold.csv",
    index=False
)


# 5. Конфигурация и всички резултати в един Joblib файл
experiment_data = {
    "experiment_name":
        "lasso_balanced_rf",
    "dataset": "GSE25055",
    "random_state": RANDOM_STATE,
    "outer_folds": 10,
    "inner_folds": 5,
    "lasso_parameter_grid": parameter_grid,
    "number_of_trees": NUMBER_OF_TREES,
    "max_depth": MAX_DEPTH,
    "min_samples_split": MIN_SAMPLES_SPLIT,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "class_weight": "balanced",
    "fold_results": fold_results,
    "comparison_table": comparison_table,
    "oof_predictions": oof_predictions,
    "selected_probes_by_fold": selected_probes_by_fold,
    "outer_splits": outer_splits
}

joblib.dump(
    experiment_data,
    RESULTS_DIR / "experiment_results.joblib"
)


print("Резултатите са запазени в:")
print(RESULTS_DIR.resolve())

display(comparison_table)

In [5]:
print("\nCustom threshold check:")
print("Threshold:", CLASSIFICATION_THRESHOLD)
print("Probability range:", custom_proba.min(), custom_proba.max())
print("Predicted pCR:", custom_pred.sum())
print("Confusion matrix:")
print(confusion_matrix(y_val, custom_pred))

print("\nProbabilities for actual pCR patients:")
print(custom_proba[y_val.to_numpy() == 1])

assert np.array_equal(
    custom_pred,
    (custom_proba >= CLASSIFICATION_THRESHOLD).astype(int)
)


Custom threshold check:
Threshold: 0.3
Probability range: 0.0 0.6133842374822649
Predicted pCR: 5
Confusion matrix:
[[23  2]
 [ 3  3]]

Probabilities for actual pCR patients:
[0.16506741 0.51063053 0.15640209 0.46291327 0.22365745 0.61338424]


In [4]:
# Прогнози и върху training пациентите.
custom_train_pred, custom_train_proba = predict_forest(
    forest,
    X_train_sel
)

print("TRAIN")
print("Реални класове:")
print(y_train.value_counts().sort_index())

print("\nПредсказани класове:")
print(
    pd.Series(custom_train_pred)
    .value_counts()
    .sort_index()
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_train_arr,
        custom_train_pred,
        labels=[0, 1]
    )
)

print(
    "Sensitivity:",
    recall_score(
        y_train_arr,
        custom_train_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_train_arr,
        custom_train_proba
    )
)


print("\nVALIDATION")
print("Реални класове:")
print(y_val.value_counts().sort_index())

print("\nCustom предсказани класове:")
print(
    pd.Series(custom_pred)
    .value_counts()
    .sort_index()
)

print("\nCustom confusion matrix:")
print(
    confusion_matrix(
        y_val,
        custom_pred,
        labels=[0, 1]
    )
)

print("\nCustom probability range:")
print(
    "Minimum:",
    custom_proba.min()
)

print(
    "Maximum:",
    custom_proba.max()
)

probability_analysis = pd.DataFrame({
    "actual_class": y_val.to_numpy(),
    "custom_probability": custom_proba,
    "custom_prediction": custom_pred,
    "sklearn_probability": sklearn_proba,
    "sklearn_prediction": sklearn_pred
})

print("\nВероятности според реалния клас:")
display(
    probability_analysis
    .groupby("actual_class")[
        [
            "custom_probability",
            "sklearn_probability"
        ]
    ]
    .agg([
        "min",
        "mean",
        "median",
        "max"
    ])
)

display(
    probability_analysis.sort_values(
        "custom_probability",
        ascending=False
    )
)

TRAIN
Реални класове:
pCR
0    224
1     51
Name: count, dtype: int64

Предсказани класове:
0    224
1     51
Name: count, dtype: int64

Confusion matrix:
[[224   0]
 [  0  51]]
Sensitivity: 1.0
ROC-AUC: 1.0

VALIDATION
Реални класове:
pCR
0    25
1     6
Name: count, dtype: int64

Custom предсказани класове:
0    30
1     1
Name: count, dtype: int64

Custom confusion matrix:
[[24  1]
 [ 6  0]]

Custom probability range:
Minimum: 0.0
Maximum: 0.5921753036678263

Вероятности според реалния клас:


custom_probability                                \
                            min      mean    median       max   
actual_class                                                    
0                      0.000000  0.142530  0.114225  0.592175   
1                      0.053847  0.234665  0.242633  0.445164   

             sklearn_probability                                
                             min      mean    median       max  
actual_class                                                    
0                       0.007426  0.238481  0.207851  0.694313  
1                       0.302802  0.444102  0.424289  0.715424

,actual_class,custom_probability,custom_prediction,sklearn_probability,sklearn_prediction
1,0,0.592175,1,0.652889,1
9,1,0.445164,0,0.365736,0
27,1,0.324519,0,0.482843,0
2,0,0.289475,0,0.463830,0
19,1,0.288163,0,0.715424,1
14,0,0.284671,0,0.694313,1
26,0,0.261581,0,0.408198,0
22,0,0.257211,0,0.264621,0
12,0,0.229068,0,0.399506,0
8,1,0.197102,0,0.302802,0


In [5]:
# ============================================================
# ЗАПАЗВАНЕ НА ПИЛОТНИЯ BASELINE ЕКСПЕРИМЕНТ
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn


# ------------------------------------------------------------
# 1. Намираме основната папка на проекта
# ------------------------------------------------------------

current_directory = Path.cwd().resolve()

if (current_directory / "src").is_dir():
    project_directory = current_directory

elif (current_directory.parent / "src").is_dir():
    project_directory = current_directory.parent

else:
    raise FileNotFoundError(
        "Не намирам основната папка на проекта."
    )


# ------------------------------------------------------------
# 2. Създаваме отделна папка за пилотния експеримент
# ------------------------------------------------------------

pilot_results_directory = (
    project_directory
    / "results"
    / "lasso_custom_rf_first_run"
)

pilot_results_directory.mkdir(
    parents=True,
    exist_ok=True
)




# ------------------------------------------------------------
# 3. Проверяваме дали са изпълнени всичките 10 folds
#
# За всеки fold имаме два реда:
# 1 ред за Custom Random Forest
# 1 ред за Sklearn Random Forest
#
# Следователно очакваме общо 20 реда.
# ------------------------------------------------------------

expected_number_of_records = 20

if len(fold_records) != expected_number_of_records:
    raise RuntimeError(
        f"Очаквахме {expected_number_of_records} записа, "
        f"но са налични {len(fold_records)}."
    )


# Проверяваме дали всеки пациент има OOF прогноза.
if custom_oof_probability.isna().any():
    raise RuntimeError(
        "Липсват Custom RF вероятности за някои пациенти."
    )

if custom_oof_prediction.isna().any():
    raise RuntimeError(
        "Липсват Custom RF прогнози за някои пациенти."
    )

if sklearn_oof_probability.isna().any():
    raise RuntimeError(
        "Липсват Sklearn RF вероятности за някои пациенти."
    )

if sklearn_oof_prediction.isna().any():
    raise RuntimeError(
        "Липсват Sklearn RF прогнози за някои пациенти."
    )

print("Проверката е успешна: всички folds са завършени.")


# ------------------------------------------------------------
# 4. Създаваме таблица с резултатите по folds
# ------------------------------------------------------------

fold_results = pd.DataFrame(
    fold_records
)


# В първоначалния код не запазихме best C във fold_records.
# Затова го добавяме от изведения резултат на експеримента.
best_c_by_fold = {
    1: 0.1,
    2: 100.0,
    3: 0.1,
    4: 100.0,
    5: 100.0,
    6: 100.0,
    7: 100.0,
    8: 100.0,
    9: 100.0,
    10: 0.1
}

fold_results["lasso_best_c"] = (
    fold_results["fold"].map(
        best_c_by_fold
    )
)


# В пилотното изпълнение не сме записали кое class_weight
# е избрано от GridSearchCV за всеки fold.
#
# Не измисляме стойности, а отбелязваме честно,
# че тази информация не е запазена.
fold_results["lasso_best_class_weight"] = (
    "not_recorded"
)


# Подреждаме редовете.
fold_results = fold_results.sort_values(
    by=[
        "fold",
        "model"
    ]
).reset_index(
    drop=True
)


# Записваме таблицата.
fold_results_path = (
    pilot_results_directory
    / "fold_results.csv"
)

fold_results.to_csv(
    fold_results_path,
    index=False
)


# ------------------------------------------------------------
# 5. Изчисляваме средна стойност и стандартно отклонение
# ------------------------------------------------------------

metric_columns = [
    "selected_probes",
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "sensitivity",
    "specificity",
    "rf_time_seconds"
]

cv_summary = (
    fold_results
    .groupby("model")[metric_columns]
    .agg([
        "mean",
        "std"
    ])
)


# Премахваме двойното ниво на имената на колоните.
cv_summary.columns = [
    column_name + "_" + statistic
    for column_name, statistic
    in cv_summary.columns
]

cv_summary = cv_summary.reset_index()


summary_path = (
    pilot_results_directory
    / "cross_validation_summary.csv"
)

cv_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 6. Създаваме таблица с OOF прогнозите
#
# OOF = out-of-fold.
# Всеки пациент е предсказан от модел, който не е бил
# обучаван с данните на този пациент.
# ------------------------------------------------------------

oof_results = pd.DataFrame({
    "patient_id": y.index.astype(str),
    "true_label": y.to_numpy(dtype=int),

    "custom_probability": (
        custom_oof_probability
        .to_numpy(dtype=float)
    ),

    "custom_prediction": (
        custom_oof_prediction
        .astype("int64")
        .to_numpy()
    ),

    "sklearn_probability": (
        sklearn_oof_probability
        .to_numpy(dtype=float)
    ),

    "sklearn_prediction": (
        sklearn_oof_prediction
        .astype("int64")
        .to_numpy()
    )
})


oof_results_path = (
    pilot_results_directory
    / "oof_predictions.csv"
)

oof_results.to_csv(
    oof_results_path,
    index=False
)


# ------------------------------------------------------------
# 7. Изчисляваме общите OOF метрики
# ------------------------------------------------------------

def calculate_complete_metrics(
    y_true,
    y_prediction,
    y_probability
):
    """
    Изчислява метриките върху всички OOF прогнози.
    """

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            y_true,
            y_probability
        ),

        "average_precision": average_precision_score(
            y_true,
            y_probability
        ),

        "accuracy": accuracy_score(
            y_true,
            y_prediction
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_prediction
        ),

        "f1": f1_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "sensitivity": (
            tp / (tp + fn)
            if (tp + fn) > 0
            else np.nan
        ),

        "specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }


true_labels = oof_results[
    "true_label"
].to_numpy()

custom_overall_metrics = calculate_complete_metrics(
    true_labels,
    oof_results["custom_prediction"].to_numpy(),
    oof_results["custom_probability"].to_numpy()
)

sklearn_overall_metrics = calculate_complete_metrics(
    true_labels,
    oof_results["sklearn_prediction"].to_numpy(),
    oof_results["sklearn_probability"].to_numpy()
)


overall_results = pd.DataFrame([
    {
        "model": "Custom Random Forest",
        **custom_overall_metrics
    },
    {
        "model": "Sklearn Random Forest",
        **sklearn_overall_metrics
    }
])


overall_results_path = (
    pilot_results_directory
    / "overall_oof_results.csv"
)

overall_results.to_csv(
    overall_results_path,
    index=False
)


# ------------------------------------------------------------
# 8. Запазваме избраните probes за всеки fold
# ------------------------------------------------------------

selected_probe_records = []

for fold, selected_probes in selected_probes_by_fold.items():

    for probe_id in selected_probes:

        selected_probe_records.append({
            "fold": fold,
            "probe_id": probe_id
        })


selected_probes_table = pd.DataFrame(
    selected_probe_records
)


selected_probes_path = (
    pilot_results_directory
    / "selected_probes_by_fold.csv"
)

selected_probes_table.to_csv(
    selected_probes_path,
    index=False
)


# ------------------------------------------------------------
# 9. Запазваме пълен checkpoint
#
# CSV файловете са удобни за разглеждане.
# Joblib checkpoint-ът позволява резултатите по-късно
# да бъдат заредени обратно в Python.
# ------------------------------------------------------------

pilot_checkpoint = {
    "experiment_name": "lasso_custom_rf_first_run",

    "description": (
        "Fold-specific LASSO followed by Custom and "
        "Sklearn Random Forest. Fixed classification "
        "threshold 0.5 and no RF class-imbalance correction."
    ),

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset_id": "GSE25055",

    "external_dataset_used": False,

    "random_state": RANDOM_STATE,

    "outer_folds": 10,
    "inner_folds": 5,

    "number_of_trees": NUMBER_OF_TREES,
    "max_depth": MAX_DEPTH,
    "min_samples_split": MIN_SAMPLES_SPLIT,

    "classification_threshold": 0.5,

    "lasso_parameter_grid": parameter_grid,

    "best_c_by_fold": best_c_by_fold,

    "lasso_best_class_weight_note": (
        "Not recorded separately during the pilot run."
    ),

    "fold_results": fold_results,
    "cv_summary": cv_summary,
    "overall_oof_results": overall_results,
    "oof_predictions": oof_results,

    "selected_probes_by_fold": (
        selected_probes_by_fold
    ),

    "outer_splits": outer_splits,

    "experiment_time_seconds": experiment_time,

    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__
}


checkpoint_path = (
    pilot_results_directory
    / "lasso_custom_rf_first_run.joblib"
)

joblib.dump(
    pilot_checkpoint,
    checkpoint_path
)


# ------------------------------------------------------------
# 10. Показваме резултата
# ------------------------------------------------------------

print("\nПилотният експеримент е запазен успешно.")

print("\nРезултати по folds:")
display(fold_results)

print("\nСредни стойности и стандартни отклонения:")
display(cv_summary)

print("\nОбщи OOF резултати:")
display(overall_results)

print("\nСъздадени файлове:")

for saved_file in sorted(
    pilot_results_directory.iterdir()
):
    print("-", saved_file.name)

Проверката е успешна: всички folds са завършени.

Пилотният експеримент е запазен успешно.

Резултати по folds:


,fold,model,selected_probes,rf_time_seconds,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,lasso_best_c,lasso_best_class_weight
0,1,Custom Random Forest,74,7.523554,0.860000,0.552673,0.774194,0.480000,0.000000,0.0,0.000000,0.960000,0.1,not_recorded
1,1,Sklearn Random Forest,74,0.043432,0.813333,0.504762,0.806452,0.563333,0.250000,0.5,0.166667,0.960000,0.1,not_recorded
2,2,Custom Random Forest,750,200.549500,0.706667,0.443754,0.806452,0.563333,0.250000,0.5,0.166667,0.960000,100.0,not_recorded
3,2,Sklearn Random Forest,750,0.079513,0.766667,0.592816,0.838710,0.583333,0.285714,1.0,0.166667,1.000000,100.0,not_recorded
4,3,Custom Random Forest,77,9.134013,0.733333,0.451703,0.741935,0.460000,0.000000,0.0,0.000000,0.920000,0.1,not_recorded
5,3,Sklearn Random Forest,77,0.047992,0.760000,0.631349,0.870968,0.666667,0.500000,1.0,0.333333,1.000000,0.1,not_recorded
6,4,Custom Random Forest,726,161.759546,0.660000,0.351706,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded
7,4,Sklearn Random Forest,726,0.079089,0.753333,0.501578,0.838710,0.583333,0.285714,1.0,0.166667,1.000000,100.0,not_recorded
8,5,Custom Random Forest,661,142.445101,0.753333,0.391880,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded
9,5,Sklearn Random Forest,661,0.094738,0.846667,0.541967,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded



Средни стойности и стандартни отклонения:


,model,selected_probes_mean,selected_probes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,...,f1_mean,f1_std,precision_mean,precision_std,sensitivity_mean,sensitivity_std,specificity_mean,specificity_std,rf_time_seconds_mean,rf_time_seconds_std
0,Custom Random Forest,525.4,310.523286,0.743689,0.103595,0.451064,0.129900,0.804086,0.029280,0.506667,...,0.053571,0.113252,0.15,0.337474,0.033333,0.070273,0.980000,0.028284,135.431495,87.951518
1,Sklearn Random Forest,525.4,310.523286,0.778989,0.060291,0.518811,0.082572,0.823441,0.032197,0.545583,...,0.165476,0.186504,0.45,0.497214,0.103333,0.119102,0.987833,0.019595,0.085458,0.027916



Общи OOF резултати:


,model,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,true_negative,false_positive,false_negative,true_positive
0,Custom Random Forest,0.748820,0.402658,0.803922,0.507504,0.062500,0.285714,0.035088,0.979920,244,5,55,2
1,Sklearn Random Forest,0.775594,0.452917,0.823529,0.546607,0.181818,0.666667,0.105263,0.987952,246,3,51,6



Създадени файлове:
- cross_validation_summary.csv
- fold_results.csv
- lasso_custom_rf_first_run.joblib
- oof_predictions.csv
- overall_oof_results.csv
- selected_probes_by_fold.csv
